# 02 — Modelling, Evaluation & Ethical AI
Steps 4–5: train and compare models, evaluate on the held-out test set, then run an explainability + bias/fairness + adversarial-robustness audit.

In [ ]:
import json, joblib, warnings, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings('ignore'); sns.set_theme(style='whitegrid')
ACCENT='#1F4E79'
tr = pd.read_parquet('data/train_processed.parquet')
te = pd.read_parquet('data/test_processed.parquet')
ytr, yte = tr['label'].values, te['label'].values
cat_te = te['attack_cat'].values
Xtr = tr.drop(columns=['label','attack_cat']); Xte = te.drop(columns=['label','attack_cat'])

## 1. Train models
Supervised: Logistic Regression, Decision Tree, Random Forest, XGBoost (all class-weight balanced for the minority attack class). Plus an unsupervised **Isolation Forest** trained on normal traffic only, to represent a pure anomaly-detection baseline.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
     roc_auc_score, average_precision_score, confusion_matrix)
spw = (ytr==0).sum()/(ytr==1).sum()
models = {
  'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
  'Decision Tree': DecisionTreeClassifier(max_depth=12, class_weight='balanced', random_state=42),
  'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=18, n_jobs=-1, class_weight='balanced', random_state=42),
  'XGBoost': XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, scale_pos_weight=spw, eval_metric='logloss', random_state=42),
}
res={}
for n,m in models.items():
    m.fit(Xtr,ytr); p=m.predict_proba(Xte)[:,1]; pred=(p>=.5).astype(int)
    res[n]=dict(recall=recall_score(yte,pred), precision=precision_score(yte,pred),
                f1=f1_score(yte,pred), roc_auc=roc_auc_score(yte,p), pr_auc=average_precision_score(yte,p))
pd.DataFrame(res).T.round(4)

## 2. Model comparison & selection
Detecting attacks is the costly-miss problem, so **recall on the attack class** is the primary metric, supported by PR-AUC (minority-aware). **Random Forest** is selected: best recall (0.974) and best PR-AUC (0.989), essentially tied on F1 with the single Decision Tree but far more robust. See figures/model_comparison.png, model_roc_pr.png, model_confusion.png.

In [ ]:
best = max(res, key=lambda n:(res[n]['recall'], res[n]['pr_auc']))
print('Selected best model:', best)
joblib.dump(models[best], 'models/best_model.joblib')

## 3. Explainability (SHAP / PDP / ICE)
Global SHAP (notebook 01) plus partial-dependence and ICE curves show *how* the top features shift attack probability — e.g. specific TTL ranges sharply raise predicted risk. See figures/ethics_pdp_ice.png.

## 4. Bias & fairness audit
Network traffic has no demographic attributes, so classic demographic parity does not apply directly. Instead we audit **detection parity across attack families** (the operationally meaningful subgroups) and across service groups. Most families are detected at ~0.99–1.00 recall, but **Fuzzers** lag at ~0.82 — a disparate-impact ratio of 0.82 (min/max). Mitigations: class-balanced resampling / targeted augmentation for Fuzzers, per-family threshold tuning, and cost-sensitive learning.

In [ ]:
fam = {}
pred = (models[best].predict_proba(Xte)[:,1]>=.5).astype(int)
for f in sorted(set(cat_te)):
    if f=='Normal': continue
    m = cat_te==f; fam[f]=round((pred[m]==1).mean(),3)
print('recall by attack family:', fam)
print('disparate impact (min/max):', round(min(fam.values())/max(fam.values()),3))

## 5. Limitations & robustness (2026 best practice)
**Overfitting:** train F1 0.98 vs test F1 0.92 (0.06 gap) — mild, acceptable. **Leakage:** avoided via per-split fitting. **Concept drift:** the train/test split is known to shift, so production use needs drift monitoring and periodic retraining. **Adversarial robustness:** under small feature-space perturbations, F1 falls 0.92→0.80 — clean accuracy does **not** imply robustness. Recommended defences: adversarial training, ensembles, and explanation-drift monitoring. See figures/ethics_adversarial.png.

In [ ]:
from sklearn.metrics import f1_score
rng=np.random.default_rng(0)
for eps in [0,0.1,0.3,0.5]:
    Xp = Xte.values + rng.normal(0,eps,Xte.shape)
    pp_=(models[best].predict_proba(Xp)[:,1]>=.5).astype(int)
    print(f'sigma={eps}: F1={f1_score(yte,pp_):.3f}')

## 6. Applied mitigations & drift evidence
**Concept drift** is *measured* with the Population Stability Index (PSI) — drift is mild here (0 major, 2 moderate, all in connection-counter features). **Fuzzers** (the weakest family, 0.82 recall) is mitigated two ways: threshold tuning lifts recall to 0.94 for a small F1 cost; SMOTE oversampling is tested but does not help — reported honestly. See src/run_5b_mitigations.py and figures/mit_drift_psi.png, mit_fuzzers.png.

In [ ]:
# concept drift via PSI (train vs test)
def psi(a,b,bins=10):
    import numpy as np
    qs=np.unique(np.quantile(a,np.linspace(0,1,bins+1))); qs[0],qs[-1]=-np.inf,np.inf
    if len(qs)<3: return 0.0
    ea=np.clip(np.histogram(a,qs)[0]/len(a),1e-4,None)
    eb=np.clip(np.histogram(b,qs)[0]/len(b),1e-4,None)
    return float(np.sum((eb-ea)*np.log(eb/ea)))
ps={c:round(psi(Xtr[c].values,Xte[c].values),3) for c in Xtr.columns}
print('most-drifted:', sorted(ps.items(), key=lambda kv:-kv[1])[:5])

In [ ]:
# Fuzzers mitigation via threshold tuning
fz = cat_te=='Fuzzers'; proba = models[best].predict_proba(Xte)[:,1]
for t in [0.5, 0.35]:
    pred=(proba>=t).astype(int)
    print(f'thr={t}: Fuzzers recall={ (pred[fz]==1).mean():.3f}, '
          f'overall F1={f1_score(yte,pred):.3f}')

## 7. Extended analysis (SVM, neural net, clustering, LIME, fairness, MLflow)
This section broadens the study: it adds a linear SVM and a neural network to the model comparison, tests the data's unsupervised structure with K-Means, produces a local LIME explanation, audits fairness with equalised odds, and logs every run to MLflow. Each block below mirrors a script in `src/`.

### 7.1 More models — linear SVM and a neural network (MLP)

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import recall_score, f1_score, average_precision_score
svm = CalibratedClassifierCV(LinearSVC(class_weight='balanced', max_iter=5000), cv=3).fit(Xtr, ytr)
mlp = MLPClassifier(hidden_layer_sizes=(64,32), early_stopping=True, max_iter=60, random_state=42).fit(Xtr, ytr)
for name, m in [('SVM (linear)', svm), ('Neural Net (MLP)', mlp)]:
    p = m.predict_proba(Xte)[:,1]; pred=(p>=.5).astype(int)
    print(f'{name}: recall={recall_score(yte,pred):.3f} f1={f1_score(yte,pred):.3f} pr_auc={average_precision_score(yte,p):.3f}')
# Result: MLP is competitive but does not beat the tree ensembles — the usual tabular-data outcome.

### 7.2 Unsupervised structure — K-Means with Elbow and Silhouette
Does the attack structure appear without labels? Only weakly — which justifies the supervised approach.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler
import numpy as np
Xz = StandardScaler().fit_transform(Xtr)
idx = np.random.RandomState(42).choice(len(Xz), 6000, replace=False)
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xz[idx])
    print(f'k={k}: inertia={km.inertia_:.0f} silhouette={silhouette_score(Xz[idx], km.labels_):.3f}')
km2 = KMeans(2, n_init=10, random_state=42).fit(Xz[idx])
print('k=2 ARI vs true labels:', round(adjusted_rand_score(ytr[idx], km2.labels_), 3), '(weak — labels matter)')

### 7.3 Local explanation with LIME
LIME explains one prediction; it should agree with global SHAP (TTL features dominate).

In [ ]:
from lime.lime_tabular import LimeTabularExplainer
import numpy as np
expl = LimeTabularExplainer(Xtr.values, feature_names=list(Xtr.columns),
                            class_names=['normal','attack'], random_state=42)
proba = models[best].predict_proba(Xte)[:,1]
i = int(np.where((yte==1)&(proba>0.95))[0][0])
exp = expl.explain_instance(Xte.values[i], models[best].predict_proba, num_features=8)
print(f'instance {i}, p(attack)={proba[i]:.2f}')
for feat, wt in exp.as_list(): print(f'  {feat:38s} {wt:+.3f}')

### 7.4 Fairness — equalised odds across service groups
Recall is even (small TPR gap), but false-positive rates diverge (large FPR gap) — an equalised-odds violation a recall-only view would miss. See src/fairness_audit.py.

### 7.5 Experiment tracking with MLflow
Every model run is logged to MLflow (params, metrics, artifacts) via `src/track_mlflow.py`, so results are versioned and searchable rather than scattered across notebooks:
```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db   # then open http://localhost:5000
```